# Leave one video out — cross-participant similarity (step 11.31)
Select **Runtime → Change runtime type → T4 GPU**. Upload the support ZIP and your rsa_class CSV to Drive, edit the paths below, then run all cells. This notebook consumes existing **Mahalanobis or correlation step-1 maps**, from result/package ZIPs or an unpacked pipeline data folder. Every unique cat1/cat2 label is omitted in turn. Pooled inputs produce pooled maps; per-run inputs produce separate runs.

The H support package uses radius 4 and the committed EmoC human cohort; the D package uses radius 3. Keep species in separate output folders. Support packages contain metadata, a mask and code; input maps must already exist.

In [ ]:
# @title Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')
%pip -q install nibabel scipy numpy
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime and rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# @title Edit these paths
PACKAGE_ZIP = '/content/drive/MyDrive/rsa_colab/pkg_stimulus_contribution_H.zip'
CLASS_CSV = '/content/drive/MyDrive/rsa_colab/class_all.csv'  # None if bundled in the package
DIS_METHOD = 'mahalanobis'  # or 'correlation'
MAH_FOLD = 'stim-wise'  # same folding as the input maps
INPUT_SOURCES = ['/content/drive/MyDrive/rsa_colab/results_EmoC_r4']
OUT_DIR = '/content/drive/MyDrive/rsa_colab/results_stimulus_contribution_H'
WORK_DIR = '/content/stimulus_work'  # local scratch, not Drive
VOXEL_BATCH = 128
COMPARISON_BATCH = 4096
FORCE = False


In [ ]:
# @title Extract support package and verify the GPU statistics
from pathlib import Path
import importlib, itertools, json, shutil, sys, zipfile
import numpy as np
PACKAGE_ROOT = Path('/content/stimulus_package')
with zipfile.ZipFile(PACKAGE_ZIP) as archive:
    for name in archive.namelist():
        target = (PACKAGE_ROOT / name).resolve()
        if not target.is_relative_to(PACKAGE_ROOT.resolve()):
            raise ValueError('Unsafe archive member: ' + name)
    archive.extractall(PACKAGE_ROOT)
sys.path.insert(0, str(PACKAGE_ROOT))
import stimulus_contribution, run_colab_stimulus_contribution
importlib.reload(stimulus_contribution)
importlib.reload(run_colab_stimulus_contribution)
pairs = list(itertools.combinations(['A1', 'A2', 'B1', 'B2', 'C1'], 2))
rng = np.random.default_rng(1131)
x = rng.integers(0, 5, (9, len(pairs))).astype(float)
y = rng.integers(0, 5, x.shape).astype(float)
x[0] = 1  # constant RDM must yield NaN
for method in ('kendall', 'spearman', 'pearson'):
    expected = stimulus_contribution.correlate_omissions(x, y, pairs, method=method)
    actual = stimulus_contribution.correlate_omissions(x, y, pairs, method=method, device='cuda', comparison_batch=7)
    np.testing.assert_allclose(actual, expected, atol=1e-12, rtol=1e-12, equal_nan=True)
print('CPU/GPU parity passed, including ties and constant RDMs.')
manifest = json.loads((PACKAGE_ROOT / 'manifest.json').read_text())
class_path = Path(CLASS_CSV) if CLASS_CSV else (PACKAGE_ROOT / manifest['class_csv'] if manifest.get('class_csv') else None)
if class_path is None:
    raise ValueError('Set CLASS_CSV to your rsa_class CSV with cat1/cat2 columns.')
manifest['pairs'] = stimulus_contribution.read_pairs(class_path)
manifest['dis_method'], manifest['mah_fold'] = DIS_METHOD, MAH_FOLD
print({k: manifest[k] for k in ('dataset', 'specie', 'radius', 'dis_method', 'rsa_method', 'pair_origin')})
print('Participants:', manifest['participants'])
print('Pairs:', len(manifest['pairs']), 'Omissions:', stimulus_contribution.validate_pairs(manifest['pairs'])[1])
for source in INPUT_SOURCES:
    if not Path(source).exists():
        raise FileNotFoundError('Edit INPUT_SOURCES: ' + source)
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
import nibabel as nib
mask = np.asarray(nib.load(str(PACKAGE_ROOT / manifest['mask'])).dataobj)
nvox = int((np.isfinite(mask) & (mask != 0)).sum())
max_sessions = max(sum(sum(int(e['run_N']) == r for e in entries) for entries in manifest['runs_by_sub'].values()) for r in {int(e['run_N']) for entries in manifest['runs_by_sub'].values() for e in entries})
if DIS_METHOD == 'mahalanobis' and MAH_FOLD != 'stim-wise-all-runs':
    max_sessions = len(manifest['participants'])
needed = max_sessions * nvox * len(manifest['pairs']) * 8
free = shutil.disk_usage(WORK_DIR).free
print(f'RDM scratch needed: {needed / 1e9:.1f} GB; available: {free / 1e9:.1f} GB')
if needed * 1.05 > free:
    raise RuntimeError('Insufficient local scratch space; choose a runtime with more disk.')


In [ ]:
# @title Run step 11.31 on the GPU (resumable)
RESULT_ZIP = run_colab_stimulus_contribution.run_package(
    str(PACKAGE_ROOT), INPUT_SOURCES, OUT_DIR, work=WORK_DIR, device='cuda',
    voxel_batch=VOXEL_BATCH, comparison_batch=COMPARISON_BATCH, force=FORCE,
    class_csv=CLASS_CSV, dis_method=DIS_METHOD, mah_fold=MAH_FOLD)
print(RESULT_ZIP)


Results persist in OUT_DIR under `EmoC/results/RSA_same/basic-block/class_all/leave-one-out/by_stim/run-NN/`. For example: `b_GreyMatter2mmB-r-4_kendall_DogA1.nii.gz`. Each map has a population standard deviation and JSON provenance. Download the completed `result_stimulus_contribution_*.zip` and merge it with `tools/unpack_results.py`.

The map is the group similarity after removing the chosen cat1/cat2 label from both target and peer RDMs, with equal participant weight. Pooled Mahalanobis inputs are processed once and saved under by_stim/pooled; other folds have a fold subfolder. Correlation and stim-wise-all-runs inputs preserve runs. It is not a subtraction from the full-data map. Reduce VOXEL_BATCH or COMPARISON_BATCH if the GPU runs out of memory. Disconnect and delete the runtime when finished.